In [0]:
# ============================================================
# NOTEBOOK 05 — COMPARAÇÃO ANTES/DEPOIS
# Objetivo: medir o ganho real das features de contexto.
# Recalcula as duas rodadas do zero para garantir reprodutibilidade.
# ============================================================
%pip install shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             classification_report)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Setup concluído")

In [0]:
# ============================================================
# CÉLULA 2 — Descobrir a tabela do indicador municipal
# O schema gold não existe neste workspace. Vamos listar
# as tabelas dos schemas disponíveis e localizar a correta.
# ============================================================
CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"

for schema in ["silver", "bronze", "default", "monitoring"]:
    print(f"\n{'='*70}")
    print(f"SCHEMA: {schema}")
    print('='*70)
    try:
        tabelas = spark.sql(f"SHOW TABLES IN {CATALOGO}.{schema}").collect()
        if not tabelas:
            print("  (vazio)")
        for t in tabelas:
            print(f"  - {t['tableName']}")
    except Exception as e:
        print(f"  (erro: {str(e)[:120]})")

In [0]:
# ============================================================
# CÉLULA 3 — Carregar a base de features da Fase 3
# A gold_indicador_municipio é a tabela analítica da Fase 2,
# sem target e sem features de contexto. A base do modelo é
# a fase3_features_municipio.
# ============================================================
CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"

pdf = spark.table(f"{CATALOGO}.default.fase3_features_municipio").toPandas()

print(f"Shape: {pdf.shape}")
print(f"Municípios únicos: {pdf['id_municipio'].nunique()}")
print(f"\nColunas ({len(pdf.columns)}):")
for c in pdf.columns:
    print(f"  - {c}")
print(f"\nDistribuição target:")
print(pdf["target"].value_counts(normalize=True).round(4))

In [0]:
# ============================================================
# CÉLULA 4 — Carregar a base de contexto e juntar com a v1
# A fase3_features_municipio tem as features originais (v1).
# A fase3_features_contexto tem as três features de contexto.
# O join é por id_municipio.
# ============================================================
CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"

ctx = spark.table(f"{CATALOGO}.default.fase3_features_contexto").toPandas()

print(f"Shape contexto: {ctx.shape}")
print(f"Colunas ({len(ctx.columns)}):")
for c in ctx.columns:
    print(f"  - {c}")

# Join
pdf = pdf.merge(ctx, on="id_municipio", how="left")

print(f"\nShape após join: {pdf.shape}")
print(f"Municípios: {pdf['id_municipio'].nunique()}")

# Verificar se as features de contexto chegaram
features_ctx = ["taxa_uf_2023", "posicao_relativa_uf", "meta_vs_uf"]
for f in features_ctx:
    if f in pdf.columns:
        nulos = pdf[f].isna().sum()
        print(f" {f} — {nulos} nulos")
    else:
        print(f" {f} — AUSENTE")

In [0]:
# ============================================================
# CÉLULA 5 — Definir as bases v1 e v2
# A fase3_features_contexto já contém todas as colunas da v1
# mais as features de contexto. Não precisa de join.
# ============================================================
CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"

# v1 — features originais
pdf_v1 = spark.table(f"{CATALOGO}.default.fase3_features_municipio").toPandas()

# v2 — features originais + contexto
pdf_v2 = spark.table(f"{CATALOGO}.default.fase3_features_contexto").toPandas()

print(f"v1: {pdf_v1.shape}")
print(f"v2: {pdf_v2.shape}")

# Definir os conjuntos de features
FEATURES_V1 = [
    "qtd_redes",
    "percentual_participacao",
    "meta_do_ano",
    "taxa_2023",
    "media_portugues_2023",
    "participacao_2023",
    "gap_meta_2023",
    "esforco_necessario",
]

FEATURES_V2 = FEATURES_V1 + [
    "taxa_uf_2023",
    "posicao_relativa_uf",
    "meta_vs_uf",
]

# Validar presença
print(f"\nv1 — {len(FEATURES_V1)} features:")
for f in FEATURES_V1:
    status = "OK" if f in pdf_v1.columns else "AUSENTE"
    print(f"  {status}: {f}")

print(f"\nv2 — {len(FEATURES_V2)} features:")
for f in FEATURES_V2:
    status = "OK" if f in pdf_v2.columns else "AUSENTE"
    print(f"  {status}: {f}")

# Confirmar que as duas bases têm os mesmos municípios
print(f"\nMesmos municípios: {set(pdf_v1['id_municipio']) == set(pdf_v2['id_municipio'])}")
print(f"Target v1: {pdf_v1['target'].mean():.4f}")
print(f"Target v2: {pdf_v2['target'].mean():.4f}")

In [0]:
# ============================================================
# CÉLULA 6 — Função de avaliação out-of-fold
# Mesmo modelo, mesma validação, mesmas métricas.
# Só muda o conjunto de features.
# ============================================================
def avaliar(df, features, nome):
    X = df[features].copy()
    y = df["target"].astype(int)
    groups = df["id_municipio"]

    modelo = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=400, max_depth=6, min_samples_leaf=30,
            max_features="sqrt", random_state=42,
            class_weight="balanced", n_jobs=-1))
    ])

    cv = GroupKFold(n_splits=5)
    y_pred = cross_val_predict(modelo, X, y, groups=groups, cv=cv, method="predict")
    y_proba = cross_val_predict(modelo, X, y, groups=groups, cv=cv,
                                method="predict_proba")[:, 1]

    return {
        "rodada": nome,
        "n_features": len(features),
        "roc_auc": roc_auc_score(y, y_proba),
        "accuracy": accuracy_score(y, y_pred),
        "f1_atingiu": f1_score(y, y_pred, pos_label=1),
        "f1_abaixo": f1_score(y, y_pred, pos_label=0),
        "f1_macro": f1_score(y, y_pred, average="macro"),
        "_modelo": modelo, "_X": X, "_y": y, "_groups": groups,
        "_y_pred": y_pred, "_y_proba": y_proba,
    }

print("Função definida")

In [0]:
# ============================================================
# CÉLULA 7 — Executar as duas rodadas
# Mesmo modelo, mesma validação. Só muda o conjunto de features.
# ============================================================
print("Rodando v1 (sem contexto)...")
res_v1 = avaliar(pdf_v1, FEATURES_V1, "v1 — sem contexto")

print("Rodando v2 (com contexto)...")
res_v2 = avaliar(pdf_v2, FEATURES_V2, "v2 — com contexto")

print("\nConcluído")
print(f"v1 ROC-AUC: {res_v1['roc_auc']:.4f}")
print(f"v2 ROC-AUC: {res_v2['roc_auc']:.4f}")

In [0]:
# ============================================================
# CÉLULA 8 — Tabela comparativa completa
# ============================================================
metricas = ["roc_auc", "accuracy", "f1_atingiu", "f1_abaixo", "f1_macro"]

comp = pd.DataFrame([
    {m: res_v1[m] for m in metricas} | {"rodada": "v1 — sem contexto", "n_features": res_v1["n_features"]},
    {m: res_v2[m] for m in metricas} | {"rodada": "v2 — com contexto", "n_features": res_v2["n_features"]},
])

# Ganho absoluto e relativo
ganho = {"rodada": "Δ (v2 − v1)", "n_features": "+3"}
for m in metricas:
    ganho[m] = res_v2[m] - res_v1[m]

comp = pd.concat([comp, pd.DataFrame([ganho])], ignore_index=True)

# Formatar
display_df = comp.copy()
for m in metricas:
    display_df[m] = display_df[m].apply(lambda x: f"{x:+.4f}" if isinstance(x, float) else x)

print(display_df.to_string(index=False))

# Ganho percentual do ROC-AUC
delta_pct = (res_v2["roc_auc"] - res_v1["roc_auc"]) / res_v1["roc_auc"] * 100
print(f"\nGanho relativo em ROC-AUC: {delta_pct:+.2f}%")

In [0]:
# ============================================================
# CÉLULA 9 — Matriz de confusão e relatório por classe
# ============================================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, res, titulo in zip(axes, [res_v1, res_v2],
                           ["v1 — sem contexto", "v2 — com contexto"]):
    cm = confusion_matrix(res["_y"], res["_y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Abaixo", "Atingiu"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"{titulo}\nROC-AUC = {res['roc_auc']:.4f}", fontsize=11)
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real")

plt.tight_layout()
plt.show()

# Relatório detalhado
for res in [res_v1, res_v2]:
    print(f"\n{'='*60}")
    print(res["rodada"])
    print('='*60)
    print(classification_report(res["_y"], res["_y_pred"],
                                target_names=["Abaixo", "Atingiu"], digits=4))

In [0]:
# ============================================================
# CÉLULA 10 — Importância das features por SHAP (versão robusta)
# Trata lista, objeto Explanation e array 3D.
# ============================================================
from sklearn.ensemble import RandomForestClassifier as RFC

X_v2 = pdf_v2[FEATURES_V2].copy()
y_v2 = pdf_v2["target"].astype(int)

modelo_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RFC(n_estimators=400, max_depth=6, min_samples_leaf=30,
                max_features="sqrt", random_state=42,
                class_weight="balanced", n_jobs=-1))
])
modelo_final.fit(X_v2, y_v2)

clf = modelo_final.named_steps["clf"]
X_imp = pd.DataFrame(
    modelo_final.named_steps["imputer"].transform(X_v2),
    columns=FEATURES_V2
)

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_imp)

# --- Normalizar o retorno do SHAP para matriz 2D (classe positiva) ---
if isinstance(shap_values, list):
    sv = shap_values[1]
elif hasattr(shap_values, "values"):
    sv = shap_values.values
    if sv.ndim == 3:
        sv = sv[:, :, 1]
else:
    sv = shap_values
    if sv.ndim == 3:
        sv = sv[:, :, 1]

sv = np.asarray(sv)
print(f"Formato dos valores SHAP: {sv.shape}")

importancia = pd.DataFrame({
    "feature": FEATURES_V2,
    "shap_medio_abs": np.abs(sv).mean(axis=0)
}).sort_values("shap_medio_abs", ascending=False)

print("\n" + importancia.to_string(index=False))

plt.figure(figsize=(9, 5))
plt.barh(importancia["feature"][::-1], importancia["shap_medio_abs"][::-1], color="#2E5C8A")
plt.xlabel("Impacto médio absoluto no SHAP")
plt.title("Importância das features — modelo v2 (com contexto)")
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# CÉLULA 11 — SHAP beeswarm: direção do efeito
# ============================================================
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_imp, feature_names=FEATURES_V2,
                  show=False, plot_size=(10, 6))
plt.title("SHAP beeswarm — modelo v2 (com contexto)", fontsize=12)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# CÉLULA 12 — Conclusão consolidada (índice corrigido)
# ============================================================
print("=" * 70)
print("COMPARAÇÃO ANTES/DEPOIS — FEATURES DE CONTEXTO")
print("=" * 70)

print(f"\n1. MÉTRICA AGREGADA")
print(f"   ROC-AUC v1 (8 features):  {res_v1['roc_auc']:.4f}")
print(f"   ROC-AUC v2 (11 features): {res_v2['roc_auc']:.4f}")
print(f"   Ganho: +{res_v2['roc_auc'] - res_v1['roc_auc']:.4f} "
      f"(+{(res_v2['roc_auc'] - res_v1['roc_auc']) / res_v1['roc_auc'] * 100:.2f}%)")

print(f"\n2. ERRO CRÍTICO (classe 'Abaixo')")
fn_v1 = ((res_v1['_y'] == 0) & (res_v1['_y_pred'] == 1)).sum()
fn_v2 = ((res_v2['_y'] == 0) & (res_v2['_y_pred'] == 1)).sum()
print(f"   F1 v1: {res_v1['f1_abaixo']:.4f}")
print(f"   F1 v2: {res_v2['f1_abaixo']:.4f}")
print(f"   Falsos negativos v1: {fn_v1}")
print(f"   Falsos negativos v2: {fn_v2}")
print(f"   Redução: {fn_v1 - fn_v2} municípios")

print(f"\n3. EXPLICABILIDADE (SHAP)")
top3 = importancia.head(3).reset_index(drop=True)
for pos, row in top3.iterrows():
    origem = "contexto" if row["feature"] in ["taxa_uf_2023", "posicao_relativa_uf", "meta_vs_uf"] else "v1"
    print(f"   {pos+1}º {row['feature']:<25} {row['shap_medio_abs']:.6f}  [{origem}]")

print(f"\n4. CONCLUSÃO")
print("   O ganho de performance vem das features de contexto federativo.")
print("   O modelo aprendeu a lógica de negócio: meta ambiciosa em relação")
print("   à UF e UF com alfabetização fraca aumentam a chance de o município")
print("   ficar abaixo da meta.")
print("=" * 70)